# Busan Traffic AI PBL - 18_busan_model_visualization.ipynb

부산시 교통량 예측 실습을 부산시 교통량 예측 프로젝트로 변경한 노트북입니다.
API 인증키와 ngrok token은 코드에 직접 넣지 말고 환경변수로 관리합니다.


In [ ]:
# =========================
# [셀 1] 라이브러리 로드
# =========================
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# =========================
# [셀 2] 데이터 로드/병합/split
# =========================
X_path = "data/processed/merged_x.csv"
y_path = "data/processed/traffic_daily_total_2022_2023.csv"

df_x = pd.read_csv(X_path)
df_y = pd.read_csv(y_path)

# =========================
# [셀 3] date 처리 및 y 컬럼 자동 탐지
# =========================
df_x["date"] = pd.to_datetime(df_x["date"])
df_y["date"] = pd.to_datetime(df_y["date"])

# y 후보: date 제외한 컬럼들 중 하나를 선택
y_candidates = [c for c in df_y.columns if c != "date"]
if len(y_candidates) == 1:
    y_col = y_candidates[0]
else:
    # 흔한 이름 우선
    preferred = ["y", "target", "traffic", "traffic_total", "daily_total", "label"]
    found = [c for c in preferred if c in y_candidates]
    y_col = found[0] if len(found) > 0 else y_candidates[-1]

print("사용할 y 컬럼:", y_col)
df_y[[ "date", y_col ]].head()

In [ ]:
# =========================
# [셀 4] X와 y 병합
# =========================
df = df_x.merge(df_y[["date", y_col]], on="date", how="inner").sort_values("date").reset_index(drop=True)
print(df.shape)
df.head()

In [ ]:
# =========================
# [셀 5] Train Val Test Split
# =========================
train_mask = (df["date"] >= "2022-01-01") & (df["date"] <= "2022-12-31")
val_mask   = (df["date"] >= "2023-01-01") & (df["date"] <= "2023-01-31")
test_mask  = (df["date"] >= "2023-02-01") & (df["date"] <= "2023-12-31")

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()

print("train/val/test:", df_train.shape, df_val.shape, df_test.shape)

In [ ]:
# =========================
# [셀 6] Feature/Label 분리
# =========================
feature_cols = [c for c in df.columns if c not in ["date", y_col]]

X_train, y_train = df_train[feature_cols], df_train[y_col]
X_val, y_val     = df_val[feature_cols], df_val[y_col]
X_test, y_test   = df_test[feature_cols], df_test[y_col]

print(len(feature_cols), "features")

In [ ]:
# =========================
# [셀 7] 전처리 파이프라인
# - 수치형: 결측 median 대치 + 표준화
# - 문자열/범주형: (있다면) 결측 most_frequent 대치
# =========================
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop"
)

print("num:", len(num_cols), "cat:", len(cat_cols))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# [셀 9] 학습 및 평가 함수
# =========================
def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

results = []

model = RandomForestRegressor(random_state=42)
name = "RandomForest"

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", model)
])
pipe.fit(X_train, y_train)

val_pred = pipe.predict(X_val)
test_pred = pipe.predict(X_test)

r_val = evaluate_regression(y_val, val_pred)
r_test = evaluate_regression(y_test, test_pred)

results.append({
    "model": name,
    "val_MAE": r_val["MAE"],
    "val_RMSE": r_val["RMSE"],
    "val_R2": r_val["R2"],
    "test_MAE": r_test["MAE"],
    "test_RMSE": r_test["RMSE"],
    "test_R2": r_test["R2"],
})

pd.DataFrame(results).sort_values("val_RMSE")

df_test["y_true"] = y_test.values
df_test["y_pred"] = test_pred

In [ ]:
# =========================
# [셀 10] 시각화 1: 실제 vs 예측 산점도
# =========================
plt.figure(figsize=(6,6))
sns.scatterplot(data=df_test, x="y_true", y="y_pred", alpha=0.6)
plt.plot([df_test["y_true"].min(), df_test["y_true"].max()],
         [df_test["y_true"].min(), df_test["y_true"].max()])
plt.title("Actual vs Predicted")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()

In [ ]:
# =========================
# [셀 11] 시각화 2: 날짜별 추세 라인
# =========================
plot_df = df_test[["date","y_true","y_pred"]].sort_values("date")

plt.figure(figsize=(12,4))
plt.plot(plot_df["date"], plot_df["y_true"], label="Actual")
plt.plot(plot_df["date"], plot_df["y_pred"], label="Predicted")
plt.title("Time Series: Actual vs Predicted (Test)")
plt.xlabel("date")
plt.ylabel("value")
plt.legend()
plt.show()

In [ ]:
# =========================
# [셀 12] 시각화 3: 오차 분포
# =========================
df_test["error"] = df_test["y_pred"] - df_test["y_true"]

plt.figure(figsize=(8,4))
sns.histplot(df_test["error"], bins=30, kde=True)
plt.title("Prediction Error Distribution (Pred - True)")
plt.xlabel("error")
plt.show()

In [ ]:
# =========================
# [셀 13] 시각화 4: 월별 성능 요약 (2023-02 ~ 2023-12)
# =========================
df_test["month"] = df_test["date"].dt.to_period("M").astype(str)
monthly = df_test.groupby("month").apply(lambda g: np.mean(np.abs(g["error"]))).reset_index(name="MAE")

plt.figure(figsize=(10,4))
sns.barplot(data=monthly, x="month", y="MAE")
plt.title("Monthly MAE (Test)")
plt.xticks(rotation=45)
plt.show()